# 04 — Anomaly Detection on AQI Time Series

**Task**: Phát hiện các giờ ô nhiễm bất thường (pollution spikes) — ví dụ cháy rừng, đốt rơm rạ, sự cố công nghiệp.

**Methods**:
1. **Rolling Z-score** (statistical baseline) — `z = (x - mean_7d) / std_7d`, flag nếu `|z| > 3`
2. **Isolation Forest** (unsupervised ML) — multivariate anomalies trên (AQI, PM2.5, PM10, weather)
3. **LSTM reconstruction error** (optional stretch) — autoencoder trên sequence, high error = anomaly

**Evaluation không có ground truth**:
- Precision-like: % anomalies được cross-validated bằng Z-score và IF (agreement rate)
- Qualitative: anomalies có align với event thực tế không (Tết, mùa đốt rơm rạ)
- Contamination rate tune qua domain knowledge (expect ~1-3% anomalies)

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
SEED = 42

## 1. Load Features

In [ ]:
FEATURES_PATH = Path('./artifacts/features_2021.parquet')
assert FEATURES_PATH.exists(), f'Run notebook 02 first to produce {FEATURES_PATH}'

feats = pd.read_parquet(FEATURES_PATH)
feats['measured_at'] = pd.to_datetime(feats['measured_at'], utc=True)
df = feats[['measured_at', 'queried_city', 'aqi', 'pm25', 'pm10',
            'temperature', 'humidity', 'pressure', 'wind',
            'aqi_roll_mean_24h', 'aqi_roll_std_24h',
            'aqi_roll_mean_7d', 'aqi_roll_std_7d']].copy()
df = df.dropna(subset=['aqi'])
print(f'Rows: {len(df):,}')
print(f'Cities: {df["queried_city"].value_counts().to_dict()}')

## 2. Method 1 — Rolling Z-Score

In [ ]:
df['zscore_7d'] = (df['aqi'] - df['aqi_roll_mean_7d']) / df['aqi_roll_std_7d'].replace(0, np.nan)
df['is_anom_zscore'] = (df['zscore_7d'].abs() > 3).astype(int)

pct = df['is_anom_zscore'].mean() * 100
print(f'Z-score anomalies: {df["is_anom_zscore"].sum():,} rows ({pct:.2f}%)')
print('\nPer-city breakdown:')
print(df.groupby('queried_city')['is_anom_zscore'].agg(['sum', 'mean']).round(4))

## 3. Method 2 — Isolation Forest (Multivariate)

In [ ]:
iforest_features = ['aqi', 'pm25', 'pm10', 'temperature', 'humidity', 'pressure', 'wind']

df_if = df.dropna(subset=iforest_features).copy()
print(f'Rows after dropna on IF features: {len(df_if):,}')

scaler = StandardScaler()
X_if = scaler.fit_transform(df_if[iforest_features])

iforest = IsolationForest(
    n_estimators=200,
    contamination=0.02,   # expect ~2% anomalies
    max_samples='auto',
    random_state=SEED,
    n_jobs=-1,
)

iforest.fit(X_if)
df_if['iforest_score'] = -iforest.score_samples(X_if)
df_if['is_anom_iforest'] = (iforest.predict(X_if) == -1).astype(int)

print(f'IsolationForest anomalies: {df_if["is_anom_iforest"].sum():,} rows '
      f'({df_if["is_anom_iforest"].mean()*100:.2f}%)')
print('\nPer-city breakdown:')
print(df_if.groupby('queried_city')['is_anom_iforest'].agg(['sum', 'mean']).round(4))

## 4. Method Comparison & Agreement

In [ ]:
cmp = df_if.merge(df[['measured_at', 'queried_city', 'is_anom_zscore']],
                  on=['measured_at', 'queried_city'], how='left')
cmp['is_anom_zscore'] = cmp['is_anom_zscore'].fillna(0).astype(int)

n_both = ((cmp['is_anom_zscore'] == 1) & (cmp['is_anom_iforest'] == 1)).sum()
n_z    = (cmp['is_anom_zscore'] == 1).sum()
n_if   = (cmp['is_anom_iforest'] == 1).sum()

print(f'Z-score only      : {n_z - n_both:,}')
print(f'IsolationForest only: {n_if - n_both:,}')
print(f'Both agree        : {n_both:,}')
if n_z > 0:
    print(f'Precision-like (Z∩IF)/Z : {n_both / n_z * 100:.1f}%')
if n_if > 0:
    print(f'Precision-like (Z∩IF)/IF: {n_both / n_if * 100:.1f}%')

cmp['anom_category'] = 'normal'
cmp.loc[(cmp['is_anom_zscore'] == 1) & (cmp['is_anom_iforest'] == 0), 'anom_category'] = 'z_only'
cmp.loc[(cmp['is_anom_zscore'] == 0) & (cmp['is_anom_iforest'] == 1), 'anom_category'] = 'if_only'
cmp.loc[(cmp['is_anom_zscore'] == 1) & (cmp['is_anom_iforest'] == 1), 'anom_category'] = 'both'

print('\nCategory distribution:')
print(cmp['anom_category'].value_counts(normalize=True).mul(100).round(2))

## 5. Visualization — Anomaly Overlay on Time Series

In [ ]:
cities_to_plot = ['ha-noi', 'ho-chi-minh-city']
cities_to_plot = [c for c in cities_to_plot if c in cmp['queried_city'].unique()][:2]

fig, axes = plt.subplots(len(cities_to_plot), 1, figsize=(16, 4.5*len(cities_to_plot)))
if len(cities_to_plot) == 1:
    axes = [axes]

for ax, city in zip(axes, cities_to_plot):
    sub = cmp[cmp['queried_city'] == city].sort_values('measured_at')
    ax.plot(sub['measured_at'], sub['aqi'], alpha=0.6, label='AQI', color='gray')

    both = sub[sub['anom_category'] == 'both']
    z_only = sub[sub['anom_category'] == 'z_only']
    if_only = sub[sub['anom_category'] == 'if_only']

    ax.scatter(both['measured_at'],   both['aqi'],   c='red',   s=30, label='Both (strong)', zorder=3)
    ax.scatter(z_only['measured_at'], z_only['aqi'], c='orange',s=20, label='Z-score only', zorder=2)
    ax.scatter(if_only['measured_at'],if_only['aqi'],c='blue',  s=20, label='IForest only', alpha=0.6, zorder=2)

    ax.set_title(f'{city} — AQI with anomalies (2021)')
    ax.set_ylabel('AQI'); ax.legend(loc='upper right')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout(); plt.show()

## 6. Contamination Rate Tuning

In [ ]:
contam_rates = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10]
tune = []
for c in contam_rates:
    m = IsolationForest(n_estimators=150, contamination=c, random_state=SEED, n_jobs=-1)
    m.fit(X_if)
    pred = (m.predict(X_if) == -1).astype(int)
    overlap = ((pred == 1) & (cmp['is_anom_zscore'].values == 1)).sum()
    tune.append({
        'contamination': c,
        'pct_flagged': pred.mean() * 100,
        'overlap_with_zscore': int(overlap),
        'overlap_pct': float(overlap / max(pred.sum(), 1) * 100),
    })

tune_df = pd.DataFrame(tune)
print(tune_df.round(2).to_string(index=False))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(tune_df['contamination'], tune_df['pct_flagged'], 'o-', color='steelblue', label='% flagged')
ax1.set_xlabel('Contamination rate'); ax1.set_ylabel('% flagged', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(tune_df['contamination'], tune_df['overlap_pct'], 's-', color='darkorange', label='overlap_pct')
ax2.set_ylabel('% overlap with Z-score', color='darkorange')
plt.title('Tuning contamination rate — trade-off')
plt.tight_layout(); plt.show()

## 7. Anomaly Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df_if['iforest_score'], bins=60, kde=True, ax=axes[0], color='steelblue')
threshold = df_if.loc[df_if['is_anom_iforest'] == 1, 'iforest_score'].min()
axes[0].axvline(threshold, color='red', linestyle='--', label=f'Anomaly threshold ≈ {threshold:.3f}')
axes[0].set_title('Isolation Forest Score Distribution')
axes[0].legend()

mask = df['zscore_7d'].notna()
sns.histplot(df.loc[mask, 'zscore_7d'], bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].axvline( 3, color='red', linestyle='--', label='+3σ')
axes[1].axvline(-3, color='red', linestyle='--')
axes[1].set_title('Rolling Z-score Distribution')
axes[1].legend()
plt.tight_layout(); plt.show()

## 8. Top Anomalies — Case Analysis

In [ ]:
top_anom = cmp[cmp['anom_category'] == 'both'].nlargest(20, 'iforest_score')[
    ['measured_at', 'queried_city', 'aqi', 'pm25', 'pm10',
     'temperature', 'humidity', 'iforest_score', 'zscore_7d']
]
print('Top 20 strongest anomalies (both methods agree):')
print(top_anom.to_string(index=False))

ha_noi_winter = cmp[(cmp['queried_city'] == 'ha-noi') &
                    (cmp['measured_at'].dt.month.isin([11, 12, 1, 2]))]
print(f'\nHa Noi winter months anomaly rate: '
      f'{ha_noi_winter["is_anom_iforest"].mean()*100:.2f}%')
summer = cmp[(cmp['queried_city'] == 'ha-noi') & (cmp['measured_at'].dt.month.isin([6, 7, 8]))]
print(f'Ha Noi summer months anomaly rate: {summer["is_anom_iforest"].mean()*100:.2f}%')

## 9. Save Model + Threshold Metadata

In [ ]:
import joblib, json

OUT_DIR = Path('./artifacts'); OUT_DIR.mkdir(exist_ok=True)
joblib.dump(iforest, OUT_DIR / 'iforest_anomaly_model.joblib')
joblib.dump(scaler,  OUT_DIR / 'iforest_scaler.joblib')

meta = {
    'model_type': 'isolation_forest',
    'feature_cols': iforest_features,
    'contamination': 0.02,
    'anomaly_score_threshold': float(threshold),
    'n_estimators': 200,
    'n_rows_trained_on': int(len(X_if)),
    'anomaly_rate_observed': float(df_if['is_anom_iforest'].mean()),
    'zscore_threshold': 3.0,
}
with open(OUT_DIR / 'iforest_anomaly_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
print('  artifacts/iforest_anomaly_model.joblib')
print('  artifacts/iforest_scaler.joblib')
print('  artifacts/iforest_anomaly_meta.json')

## 10. Production Plan

1. **Batch scoring daily** qua SageMaker Batch Transform (`lamdas/ml_inference/`)
2. **Output table**: `gold_aqi_anomalies` — `city, timestamp, aqi, score, is_anomaly, anomaly_type`
3. **Alert rule**: nếu `is_anomaly=1` AND `aqi > 150` → push SNS alert
4. **Drift monitoring**: retrain IsolationForest khi `anomaly_rate_observed` lệch > 50% so với baseline
5. **Method combination**: production sẽ dùng **cả 2** — `anomaly_type` = `both` là high-confidence, `z_only` hoặc `if_only` là medium

SageMaker training script: `sagemaker/anomaly/train.py`